# 9.2 Stage 2: Raise Arm (Expanded)

Interpolates joints to raise the arm.

---

```python
elif t < 2*d:
    self.log_stage(2, "Raising left arm")

    r = (t - d) / d
    for i, j in enumerate(self.joints):
        self.low_cmd.motor_cmd[j].q = self.interp(self.initial_pose[i], self.target_raise[i], r)
        self.low_cmd.motor_cmd[j].dq = 0
        self.low_cmd.motor_cmd[j].kp = self.kp
        self.low_cmd.motor_cmd[j].kd = self.kd
        self.low_cmd.motor_cmd[j].tau = 0
```

---

## 🧠 Big Picture: What Is This Stage Doing?

This stage tells the robot:

```text
"Move from your current pose to a raised-arm pose smoothly over time"
```

---

## ⏱️ When Does This Stage Run?

This condition:

```python
elif t < 2*d:
```

means:

```text
From time d → 2d
```

If `d = 3.0s`, then:

```text
Stage 2 runs from 3s → 6s
```

---

## 🔄 Step 1: Compute Interpolation Ratio

```python
r = (t - d) / d
```

---

### Why subtract `d`?

Because Stage 2 starts at time `t = d`.

---

### Behavior of `r`:

| Time (t) | r   |
| -------- | --- |
| t = d    | 0.0 |
| t = 1.5d | 0.5 |
| t = 2d   | 1.0 |

---

### 🧠 Interpretation

```text
r represents progress through the motion
```

---

## 🦾 Step 2: Loop Through Joints

```python
for i, j in enumerate(self.joints):
```

---

### Meaning:

* `i` → index into trajectory arrays
* `j` → actual motor index

---

### Example mapping:

| i   | j (motor index) | Joint         |
| --- | --------------- | ------------- |
| 0   | 15              | ShoulderPitch |
| 1   | 16              | ShoulderRoll  |
| ... | ...             | ...           |

---

## 🎯 Step 3: Compute Target Position

```python
self.low_cmd.motor_cmd[j].q = self.interp(self.initial_pose[i], self.target_raise[i], r)
```

---

### This is the KEY line

It computes:

[
q_{target} = (1 - r),q_{initial} + r,q_{target}
]

---

### Interpretation:

```text
Start at initial pose → gradually move toward target_raise
```

---

### At different times:

| r   | Behavior        |
| --- | --------------- |
| 0   | At initial pose |
| 0.5 | Halfway         |
| 1   | At target pose  |

---

## 🧠 Why Use `initial_pose`?

Because:

```text
The robot may start in ANY configuration
```

---

### This ensures:

* Smooth motion from actual position
* No sudden jumps
* Safe trajectory

---

## 🎯 What Is `target_raise`?

```python
self.target_raise = [-0.3, 0.2, 0.0, -1.0, 0.0, 0.5, 0.0]
```

---

### This is a **joint-space pose**

Each value corresponds to:

| Joint         | Meaning          |
| ------------- | ---------------- |
| ShoulderPitch | lift arm forward |
| ShoulderRoll  | move outward     |
| ShoulderYaw   | rotate arm       |
| Elbow         | bend             |
| Wrist         | orient hand      |

---

### 🧠 Important

These are **angles in radians**, not Cartesian positions.

---

## ⚙️ Step 4: Set Velocity Target

```python
dq = 0
```

---

### Meaning:

```text
We are not explicitly controlling velocity
```

Velocity emerges from:

* changing position targets
* PD controller response

---

## ⚙️ Step 5: Apply PD Gains

```python
kp = self.kp
kd = self.kd
```

---

### This activates:

[
\tau = K_p (q_{target} - q) + K_d (0 - \dot{q})
]

---

### Interpretation:

* Position error → drives motion
* Velocity damping → smooths motion

---

## ⚙️ Step 6: No Feedforward Torque

```python
tau = 0
```

---

### Meaning:

```text
No gravity compensation or model-based torque
```

---

### All motion is generated by:

> PD control alone

---

## 🔄 What Happens Over Time?

Each control loop iteration:

```text
1. r increases slightly
2. q_target moves closer to final value
3. PD controller generates torque
4. joints move
```

---

### Result:

```text
Smooth continuous arm motion
```

---

## 🧠 Control Theory Interpretation

This stage implements:

> **Trajectory tracking using a PD controller**

---

### Pipeline:

```text
Desired trajectory → PD control → torque → motion
```

---

## ⚠️ Important Insight: No Explicit Trajectory Planner

You are NOT using:

* inverse kinematics
* motion planning
* dynamics model

---

Instead:

```text
Simple linear interpolation in joint space
```

---

### Yet it works because:

* Motion is slow
* Gains are tuned
* Robot is compliant (QDD)

---

## 🦾 Physical Interpretation

The robot behaves like:

```text
Each joint = spring moving toward a moving target
```

---

As the target moves:

* the spring pulls the joint along
* damping prevents oscillation

---

## ⚠️ What If You Removed Interpolation?

If you did:

```python
cmd.q = target_raise
```

---

### Result:

```text
Large instantaneous error
→ Large torque spike
→ Jerky / unsafe motion
```

---

## 🤖 RL Interpretation

This stage is essentially a **scripted policy**:

```text
state → time-based function → action
```

---

### In RL, you would replace:

```python
cmd.q = interp(...)
```

with:

```python
cmd.q = policy(state)
```

---

### But note:

Even RL often needs:

* smoothing
* interpolation
* filtering

---

## 🔥 Hidden Engineering Insight

This stage demonstrates:

> How to safely move a real robot using only:

* position targets
* PD gains

---

This is one of the simplest **yet most powerful control strategies**.

---

## 🔄 Analogy

Think of this like:

```text
Slowly raising your arm over 3 seconds
```

* You don’t jump instantly
* You move gradually
* Muscles continuously adjust

---

## 🚀 Summary

This stage:

| Step            | Role                       |
| --------------- | -------------------------- |
| Compute `r`     | Motion progress            |
| Interpolate     | Generate smooth trajectory |
| Set `q`         | Target position            |
| Apply PD        | Generate torque            |
| Repeat at 50 Hz | Continuous motion          |

---

### Result:

```text
Smooth, controlled raising of the robot arm
```

---

> 🔥 This is the first stage where **code becomes visible motion**, making it one of the most important teaching sections.

